# Notebook 09 — Classical Machine Learning

## Objective

This notebook develops supervised classical machine-learning models for
respiratory disease classification using the handcrafted acoustic features
selected in Notebook 07.

The objective is to establish strong classical machine-learning baselines
before developing the deep-learning models.

The following models will be evaluated:

1. Logistic Regression
2. Support Vector Machine (SVM)
3. Random Forest

The models will be trained using the selected feature set and evaluated on
the official test split.

The primary classification task is the prediction of the eight disease
classes:

- COPD
- Healthy
- URTI
- Bronchiectasis
- Pneumonia
- Bronchiolitis
- LRTI
- Asthma

Model performance will be evaluated using accuracy, balanced accuracy,
precision, recall, F1-score, confusion matrix, and multiclass ROC-AUC where
applicable.

In [26]:

required_objects = [
    "PROJECT_ROOT",
    "train_df",
    "test_df",
    "selected_features_df",
    "selected_feature_names",
    "X_train",
    "X_test",
    "scaler",
    "X_train_scaled",
    "X_test_scaled",
    "label_encoder",
    "y_train",
    "y_test",
    "logistic_model",
    "svm_model",
    "rf_model",
]

status = []

for name in required_objects:
    exists = name in globals()
    obj = globals().get(name) if exists else None

    status.append({
        "variable": name,
        "available": exists,
        "type": type(obj).__name__ if exists else "NOT AVAILABLE",
    })

import pandas as pd
display(pd.DataFrame(status))


,variable,available,type
0,PROJECT_ROOT,True,PosixPath
1,train_df,True,DataFrame
2,test_df,True,DataFrame
3,selected_features_df,True,DataFrame
4,selected_feature_names,True,list
5,X_train,True,DataFrame
6,X_test,True,DataFrame
7,scaler,True,StandardScaler
8,X_train_scaled,True,ndarray
9,X_test_scaled,True,ndarray


In [27]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Imports completed.")

Imports completed.


In [28]:
PROJECT_ROOT = Path("/Users/vs/Sem 5/Respiratory-Disease-Classification")

FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "respiratory_cycle_features.csv"
)

SELECTED_FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_selection"
    / "selected_features.csv"
)

RECORDINGS_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "recordings.csv"
)

features_df = pd.read_csv(FEATURES_PATH)
selected_features_df = pd.read_csv(SELECTED_FEATURES_PATH)
recordings_df = pd.read_csv(RECORDINGS_PATH)

print("Feature dataset shape:", features_df.shape)
print("Selected features shape:", selected_features_df.shape)
print("Recordings shape:", recordings_df.shape)

print("\nSelected feature columns:")
print(selected_features_df.columns.tolist())

Feature dataset shape: (6898, 52)
Selected features shape: (33, 1)
Recordings shape: (920, 7)

Selected feature columns:
['feature']


In [29]:
features_with_split = features_df.merge(
    recordings_df[
        [
            "patient_id",
            "recording_id",
            "chest_location",
            "official_split"
        ]
    ],
    on=[
        "patient_id",
        "recording_id",
        "chest_location"
    ],
    how="left",
    validate="many_to_one"
)

print("Merged dataset shape:", features_with_split.shape)

print(
    "Missing official split values:",
    features_with_split["official_split"].isna().sum()
)

print("\nOfficial split counts:")
print(features_with_split["official_split"].value_counts())

Merged dataset shape: (6898, 53)
Missing official split values: 0

Official split counts:
official_split
train    4142
test     2756
Name: count, dtype: int64


In [30]:
train_df = features_with_split[
    features_with_split["official_split"] == "train"
].copy()

test_df = features_with_split[
    features_with_split["official_split"] == "test"
].copy()

print("Training cycles:", len(train_df))
print("Testing cycles :", len(test_df))

print("\nTraining disease distribution:")
print(train_df["diagnosis"].value_counts())

print("\nTesting disease distribution:")
print(test_df["diagnosis"].value_counts())

Training cycles: 4142
Testing cycles : 2756

Training disease distribution:
diagnosis
COPD              3337
Pneumonia          285
URTI               171
Healthy            147
Bronchiectasis      90
Bronchiolitis       74
LRTI                32
Asthma               6
Name: count, dtype: int64

Testing disease distribution:
diagnosis
COPD              2409
Healthy            175
Bronchiolitis       86
URTI                72
Bronchiectasis      14
Name: count, dtype: int64


In [31]:
for name in [
    "PROJECT_ROOT",
    "train_df",
    "test_df",
    "selected_features_df",
]:
    print(name, name in globals())

PROJECT_ROOT True
train_df True
test_df True
selected_features_df True


In [32]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["diagnosis"])
y_test = label_encoder.transform(test_df["diagnosis"])

class_names = label_encoder.classes_

print("Disease classes:")
print(class_names)

print("\nNumber of classes:", len(class_names))

print("\nEncoded training labels:")
print(np.bincount(y_train))

print("\nEncoded testing labels:")
print(np.bincount(y_test))

Disease classes:
['Asthma' 'Bronchiectasis' 'Bronchiolitis' 'COPD' 'Healthy' 'LRTI'
 'Pneumonia' 'URTI']

Number of classes: 8

Encoded training labels:
[   6   90   74 3337  147   32  285  171]

Encoded testing labels:
[   0   14   86 2409  175    0    0   72]


In [33]:
selected_feature_names = selected_features_df["feature"].tolist()

X_train = train_df[selected_feature_names].copy()
X_test = test_df[selected_feature_names].copy()

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("Number of selected features:", len(selected_feature_names))

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Test :", X_test.isna().sum().sum())

print("\nInfinite values:")
print("Train:", np.isinf(X_train.to_numpy()).sum())
print("Test :", np.isinf(X_test.to_numpy()).sum())

X_train shape: (4142, 33)
X_test shape : (2756, 33)
Number of selected features: 33

Missing values:
Train: 0
Test : 0

Infinite values:
Train: 0
Test : 0


In [34]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape :", X_test_scaled.shape)

print("\nTraining scaled mean:")
print(np.mean(X_train_scaled, axis=0)[:10])

print("\nTraining scaled standard deviation:")
print(np.std(X_train_scaled, axis=0)[:10])

Scaled training shape: (4142, 33)
Scaled testing shape : (2756, 33)

Training scaled mean:
[-2.95058789e-16  3.77400777e-17  7.71956135e-17  3.77400777e-17
  3.08782454e-17  5.14637423e-17  1.26943898e-16 -1.13220233e-16
 -1.08073859e-16 -3.43091616e-17]

Training scaled standard deviation:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [35]:
logistic_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)

logistic_predictions = logistic_model.predict(X_test_scaled)
logistic_probabilities = logistic_model.predict_proba(X_test_scaled)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [36]:
svm_model = SVC(
    kernel="rbf",
    class_weight="balanced",
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

svm_predictions = svm_model.predict(X_test_scaled)
svm_probabilities = svm_model.predict_proba(X_test_scaled)

print("SVM training completed.")

/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM training completed.


In [37]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)

print("Random Forest training completed.")

Random Forest training completed.


In [38]:

required = [
    "train_df",
    "test_df",
    "selected_features_df",
    "selected_feature_names",
    "scaler",
    "label_encoder",
    "logistic_model",
    "svm_model",
    "rf_model",
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"Missing variables: {missing}")

print("Training objects are available.")
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Selected features:", len(selected_feature_names))
print("Classes:", list(label_encoder.classes_))

assert len(train_df) == 4142, "Unexpected training row count"
assert len(test_df) == 2756, "Unexpected test row count"
assert len(selected_feature_names) == 33, "Unexpected feature count"


Training objects are available.
Training rows: 4142
Test rows: 2756
Selected features: 33
Classes: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']


In [39]:

from pathlib import Path
import json
import joblib
import sklearn
import pandas as pd

# Resolve the project root using the existing notebook variable.
project_root = Path(PROJECT_ROOT)
save_dir = project_root / "models" / "traditional_ml"
save_dir.mkdir(parents=True, exist_ok=True)

# Save the fitted estimators.
models_to_save = {
    "logistic_regression": logistic_model,
    "svm": svm_model,
    "random_forest": rf_model,
}

for model_name, model in models_to_save.items():
    output_path = save_dir / f"{model_name}.joblib"
    joblib.dump(model, output_path)
    print(f"Saved {model_name}: {output_path}")

# Save the shared preprocessing objects.
joblib.dump(scaler, save_dir / "scaler.joblib")
joblib.dump(label_encoder, save_dir / "label_encoder.joblib")

# Save feature order and metadata.
metadata = {
    "project": "Respiratory Disease Classification",
    "source_notebook": "notebooks/09_classical_ml.ipynb",
    "training_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "feature_count": int(len(selected_feature_names)),
    "selected_feature_names": list(selected_feature_names),
    "class_names": [str(c) for c in label_encoder.classes_],
    "scaler_used": {
        "logistic_regression": True,
        "svm": True,
        "random_forest": False,
    },
    "training_split": "Existing official_split; unchanged",
    "sklearn_version": sklearn.__version__,
    "models": {
        "logistic_regression": {
            "file": "logistic_regression.joblib",
            "class": type(logistic_model).__name__,
            "parameters": logistic_model.get_params(),
        },
        "svm": {
            "file": "svm.joblib",
            "class": type(svm_model).__name__,
            "parameters": svm_model.get_params(),
        },
        "random_forest": {
            "file": "random_forest.joblib",
            "class": type(rf_model).__name__,
            "parameters": rf_model.get_params(),
        },
    },
}

with open(save_dir / "classical_ml_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

# Save the ordered feature list separately for convenient inspection.
with open(save_dir / "selected_features.json", "w") as f:
    json.dump(list(selected_feature_names), f, indent=2)

print("\nSaved preprocessing and metadata.")
print("Output directory:", save_dir)
print("Files:")
for path in sorted(save_dir.iterdir()):
    if path.is_file():
        print(f"  {path.name} ({path.stat().st_size:,} bytes)")


Saved logistic_regression: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/traditional_ml/logistic_regression.joblib
Saved svm: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/traditional_ml/svm.joblib
Saved random_forest: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/traditional_ml/random_forest.joblib

Saved preprocessing and metadata.
Output directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/traditional_ml
Files:
  .gitkeep (0 bytes)
  classical_ml_metadata.json (2,959 bytes)
  label_encoder.joblib (558 bytes)
  logistic_regression.joblib (3,079 bytes)
  random_forest.joblib (22,986,457 bytes)
  scaler.joblib (2,111 bytes)
  selected_features.json (597 bytes)
  svm.joblib (909,191 bytes)


In [40]:

import numpy as np
import joblib

saved_logistic = joblib.load(save_dir / "logistic_regression.joblib")
saved_svm = joblib.load(save_dir / "svm.joblib")
saved_rf = joblib.load(save_dir / "random_forest.joblib")

saved_scaler = joblib.load(save_dir / "scaler.joblib")
saved_encoder = joblib.load(save_dir / "label_encoder.joblib")

# Preserve the training-time feature representations.
X_test_raw = X_test
X_test_scaled_saved = saved_scaler.transform(X_test_raw)

checks = {
    "Logistic Regression": np.array_equal(
        saved_logistic.predict(X_test_scaled_saved),
        logistic_model.predict(X_test_scaled),
    ),
    "SVM": np.array_equal(
        saved_svm.predict(X_test_scaled_saved),
        svm_model.predict(X_test_scaled),
    ),
    "Random Forest": np.array_equal(
        saved_rf.predict(X_test_raw),
        rf_model.predict(X_test),
    ),
}

assert np.array_equal(
    saved_encoder.classes_, label_encoder.classes_
), "Label encoder classes do not match"

assert all(checks.values()), f"Prediction mismatch: {checks}"

print("PASS — saved models reload correctly.")
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name} predictions match")

print("PASS — scaler and label encoder reload correctly.")


PASS — saved models reload correctly.
PASS — Logistic Regression predictions match
PASS — SVM predictions match
PASS — Random Forest predictions match
PASS — scaler and label encoder reload correctly.


In [ ]:
def evaluate_model(
    model_name,
    y_true,
    y_pred,
    y_prob,
    class_names
):
    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }

    try:
        results["roc_auc_ovr_macro"] = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )
    except ValueError:
        results["roc_auc_ovr_macro"] = np.nan

    print(f"\n{'=' * 60}")
    print(model_name)
    print(f"{'=' * 60}")

    for key, value in results.items():
        if key != "model":
            print(f"{key}: {value:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            zero_division=0
        )
    )

    return results

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

def evaluate_model(
    model_name,
    y_true,
    y_pred,
    y_prob,
    class_names
):
    all_labels = np.arange(len(class_names))

    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true,
            y_pred,
            labels=all_labels,
            average="macro",
            zero_division=0
        ),
        "macro_recall": recall_score(
            y_true,
            y_pred,
            labels=all_labels,
            average="macro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=all_labels,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }

    try:
        results["roc_auc_ovr_macro"] = roc_auc_score(
            y_true,
            y_prob,
            labels=all_labels,
            multi_class="ovr",
            average="macro"
        )
    except ValueError:
        results["roc_auc_ovr_macro"] = np.nan

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    for key, value in results.items():
        if key != "model":
            print(f"{key}: {value:.4f}")

    print("\nClassification Report:")

    print(
        classification_report(
            y_true,
            y_pred,
            labels=all_labels,
            target_names=class_names,
            zero_division=0
        )
    )

    return results

In [ ]:
logistic_results = evaluate_model(
    "Logistic Regression",
    y_test,
    logistic_predictions,
    logistic_probabilities,
    class_names
)

svm_results = evaluate_model(
    "SVM",
    y_test,
    svm_predictions,
    svm_probabilities,
    class_names
)

rf_results = evaluate_model(
    "Random Forest",
    y_test,
    rf_predictions,
    rf_probabilities,
    class_names
)

In [ ]:
# ============================================================
# NOTEBOOK 09 — STEP 2
# CLASSICAL ML MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame([
    logistic_results,
    svm_results,
    rf_results
])

results_df = results_df[
    [
        "model",
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
        "roc_auc_ovr_macro"
    ]
].sort_values(
    by="macro_f1",
    ascending=False
).reset_index(drop=True)

print("=" * 80)
print("CLASSICAL MACHINE LEARNING MODEL COMPARISON")
print("=" * 80)

display(
    results_df.style.format({
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "macro_precision": "{:.4f}",
        "macro_recall": "{:.4f}",
        "macro_f1": "{:.4f}",
        "weighted_f1": "{:.4f}",
        "roc_auc_ovr_macro": "{:.4f}"
    })
)

print("\nTest-set disease support:")
print(
    test_df["diagnosis"]
    .value_counts()
    .reindex(class_names, fill_value=0)
)

In [ ]:
# ============================================================
# NOTEBOOK 09 — STEP 3
# CONFUSION MATRICES AND PER-DISEASE PERFORMANCE
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

predictions_dict = {
    "Logistic Regression": logistic_predictions,
    "SVM": svm_predictions,
    "Random Forest": rf_predictions
}

# ------------------------------------------------------------
# 1. Confusion matrices
# ------------------------------------------------------------

for model_name, predictions in predictions_dict.items():

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=np.arange(len(class_names))
    )

    plt.figure(figsize=(9, 7))

    plt.imshow(cm, aspect="auto")
    plt.colorbar()

    plt.xticks(
        range(len(class_names)),
        class_names,
        rotation=45,
        ha="right"
    )

    plt.yticks(
        range(len(class_names)),
        class_names
    )

    plt.xlabel("Predicted Disease")
    plt.ylabel("True Disease")
    plt.title(f"{model_name} — Confusion Matrix")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center"
            )

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 2. Per-disease metrics for all models
# ------------------------------------------------------------

per_class_rows = []

for model_name, predictions in predictions_dict.items():

    report = classification_report(
        y_test,
        predictions,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    for disease in class_names:

        per_class_rows.append({
            "model": model_name,
            "disease": disease,
            "precision": report[disease]["precision"],
            "recall": report[disease]["recall"],
            "f1_score": report[disease]["f1-score"],
            "support": int(report[disease]["support"])
        })

per_class_results_df = pd.DataFrame(per_class_rows)

print("=" * 80)
print("PER-DISEASE CLASSIFICATION PERFORMANCE")
print("=" * 80)

display(
    per_class_results_df.style.format({
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1_score": "{:.4f}"
    })
)


# ------------------------------------------------------------
# 3. Compare macro F1 by disease
# ------------------------------------------------------------

f1_comparison = per_class_results_df.pivot(
    index="disease",
    columns="model",
    values="f1_score"
)

print("\n" + "=" * 80)
print("F1-SCORE BY DISEASE")
print("=" * 80)

display(
    f1_comparison.style.format("{:.4f}")
)

In [ ]:
# ============================================================
# NOTEBOOK 09 — STEP 4
# RANDOM FOREST FEATURE IMPORTANCE
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. EXTRACT RANDOM FOREST FEATURE IMPORTANCE
# ------------------------------------------------------------

rf_importance_df = pd.DataFrame({
    "feature": selected_feature_names,
    "importance": rf_model.feature_importances_
})

rf_importance_df = rf_importance_df.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 2. DISPLAY ALL FEATURES
# ------------------------------------------------------------

print("=" * 80)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 80)

display(
    rf_importance_df.style.format({
        "importance": "{:.6f}"
    })
)


# ------------------------------------------------------------
# 3. TOP 15 FEATURES
# ------------------------------------------------------------

top_15_rf = rf_importance_df.head(15)

print("\n" + "=" * 80)
print("TOP 15 RANDOM FOREST FEATURES")
print("=" * 80)

display(
    top_15_rf.style.format({
        "importance": "{:.6f}"
    })
)


# ------------------------------------------------------------
# 4. PLOT TOP 15 FEATURES
# ------------------------------------------------------------

plot_df = top_15_rf.sort_values(
    by="importance",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_df["feature"],
    plot_df["importance"]
)

plt.xlabel("Random Forest Importance")
plt.ylabel("Feature")
plt.title("Top 15 Random Forest Feature Importances")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5. FEATURE GROUP SUMMARY
# ------------------------------------------------------------

def get_feature_group(feature):

    if feature.startswith("mfcc_"):
        return "MFCC"

    if feature.startswith("wavelet_"):
        return "Wavelet"

    if feature.startswith("band_energy_"):
        return "Band Energy"

    if feature in [
        "mean",
        "std",
        "variance",
        "rms",
        "max",
        "min",
        "peak_to_peak",
        "median",
        "skewness",
        "kurtosis",
        "zero_crossing_rate",
        "energy"
    ]:
        return "Time Domain"

    if feature in [
        "spectral_centroid",
        "spectral_bandwidth",
        "spectral_rolloff",
        "spectral_flatness",
        "spectral_energy",
        "dominant_frequency",
        "spectral_entropy"
    ]:
        return "Spectral"

    return "Other"


rf_importance_df["feature_group"] = (
    rf_importance_df["feature"]
    .apply(get_feature_group)
)

group_importance = (
    rf_importance_df
    .groupby("feature_group")["importance"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
)

print("\n" + "=" * 80)
print("RANDOM FOREST IMPORTANCE BY FEATURE GROUP")
print("=" * 80)

display(
    group_importance.style.format({
        "sum": "{:.6f}",
        "mean": "{:.6f}"
    })
)

In [ ]:
# ============================================================
# NOTEBOOK 09 — STEP 5
# SAVE ALL CLASSICAL ML RESULTS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. CREATE OUTPUT DIRECTORY
# ------------------------------------------------------------

CLASSICAL_ML_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "classical_ml"
)

CLASSICAL_ML_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. SAVE MODEL COMPARISON
# ------------------------------------------------------------

results_df.to_csv(
    CLASSICAL_ML_DIR / "classical_ml_model_comparison.csv",
    index=False
)


# ------------------------------------------------------------
# 3. SAVE PER-DISEASE RESULTS
# ------------------------------------------------------------

per_class_results_df.to_csv(
    CLASSICAL_ML_DIR / "classical_ml_per_disease_results.csv",
    index=False
)


# ------------------------------------------------------------
# 4. SAVE RANDOM FOREST FEATURE IMPORTANCE
# ------------------------------------------------------------

rf_importance_df.to_csv(
    CLASSICAL_ML_DIR / "random_forest_feature_importance.csv",
    index=False
)


# ------------------------------------------------------------
# 5. SAVE FEATURE-GROUP IMPORTANCE
# ------------------------------------------------------------

group_importance.reset_index().to_csv(
    CLASSICAL_ML_DIR / "random_forest_feature_group_importance.csv",
    index=False
)


# ------------------------------------------------------------
# 6. SAVE SELECTED FEATURES USED BY THE MODELS
# ------------------------------------------------------------

pd.DataFrame({
    "feature": selected_feature_names
}).to_csv(
    CLASSICAL_ML_DIR / "classical_ml_selected_features.csv",
    index=False
)


# ------------------------------------------------------------
# 7. SAVE MODEL PREDICTIONS
# ------------------------------------------------------------

prediction_df = pd.DataFrame({
    "patient_id": test_df["patient_id"].values,
    "recording_id": test_df["recording_id"].values,
    "chest_location": test_df["chest_location"].values,
    "true_diagnosis": test_df["diagnosis"].values,

    "logistic_regression_prediction":
        label_encoder.inverse_transform(logistic_predictions),

    "svm_prediction":
        label_encoder.inverse_transform(svm_predictions),

    "random_forest_prediction":
        label_encoder.inverse_transform(rf_predictions)
})

prediction_df.to_csv(
    CLASSICAL_ML_DIR / "classical_ml_predictions.csv",
    index=False
)


# ------------------------------------------------------------
# 8. FINAL VALIDATION
# ------------------------------------------------------------

saved_files = sorted(
    CLASSICAL_ML_DIR.glob("*.csv")
)

print("=" * 80)
print("NOTEBOOK 09 — CLASSICAL ML RESULTS SAVED")
print("=" * 80)

for file_path in saved_files:
    print(file_path.name)

print("\nOutput directory:")
print(CLASSICAL_ML_DIR)

print("\nModel comparison:")
display(results_df)

print("\nBest model by macro F1:")
best_model = results_df.iloc[0]

print("Model:", best_model["model"])
print("Macro F1:", round(best_model["macro_f1"], 4))
print("Balanced Accuracy:", round(
    best_model["balanced_accuracy"], 4
))
print("Accuracy:", round(
    best_model["accuracy"], 4
))

print("\n" + "=" * 80)
print("NOTEBOOK 09 VALIDATION COMPLETE")
print("=" * 80)

## Classical Machine Learning Interpretation

Three classical machine-learning models were evaluated using the selected
handcrafted respiratory-cycle features: Logistic Regression, SVM, and Random
Forest.

Random Forest achieved the highest overall accuracy (0.8745) and macro F1
(0.2796), while SVM achieved the highest balanced accuracy (0.4583).

The difference between accuracy and macro-averaged metrics demonstrates the
strong class imbalance in the cycle-level test data. COPD dominates the test
set, while Asthma, LRTI, and Pneumonia have no test cycles in the official
cycle-level split.

Therefore, the high accuracy should not be interpreted as uniformly strong
performance across all disease classes. Per-disease results and confusion
matrices show substantial differences between classes.

Random Forest feature importance showed that MFCC features had the largest
collective importance (0.4646), followed by spectral features (0.2469) and
time-domain features (0.1860). Spectral entropy was the most important
individual feature.

These classical models provide the baseline against which the subsequent
deep-learning approaches will be compared. More reliable disease-level
assessment will be performed later using patient-level evaluation.